# IR Lab Tutorial: Web Crawling with Scrapy

In this tutorial, we build a small web crawler with [Scrapy](https://scrapy.org/), to crawl a small focused web document collection.

Learnings:

- Choosing appropriate seed URLs
- Polite crawling settings
- Extracting title and main content
- Add links to the crawler frontier
- Post processing and clean up

**Note:** Our web crawler runs actual request to a real website, thereby using resources on the requested webpages.
Crawl responsibly: Only crawl websites that allow it, keep your crawl small, and identify yourself. In our default example, we crawl [Quotes to Scrape](https://quotes.toscrape.com/), a sandbox website made for practicing web scraping. Replace it with websites that fit your own topic.

### Preparation: Install dependencies

We need [Scrapy](https://scrapy.org/) (version 2.15 or newer) to crawl, [Resiliparse](https://resiliparse.chatnoir.eu/) to extract text from HTML, and [pandas](https://pandas.pydata.org/) to inspect and clean the crawled documents.

In [1]:
# This is only needed in Google Colab, in a dev container, everything should be installed already
!pip3 install "scrapy>=2.15" resiliparse pandas requests

### Step 1: Plan your document collection

Before crawling, decide where to start crawling, and think ahead what the crawled collection will likely contain. In the later stages of the course, we will come up information needs for the crawled document collections and need to judge which documents are relevant. A good document collection therefore has a clear topic and many text-rich pages about it:

- Choose a short, descriptive name, e.g., "Hiking Trails in Thuringia" or "Vegan Recipes". (Avoid generic names like "Our Crawl" or your group name.) → `COLLECTION_NAME`
- Describe your collection in a few sentences → `COLLECTION_DESCRIPTION`
    - What kind of pages are contained (e.g., recipes, news articles, product descriptions, forum posts)?
    - Where do they come from (which websites)?
    - Which language(s) are documents written in?
- Choose websites to start the crawling → `SEED_URLS`
- Restrict which links your crawler should follow, e.g., allowed domains → `ALLOW_DOMAINS`
- Limit how much your crawler crawls → `MAX_PAGES`
- Identify your crawler, telling maintainers how to contact you → `USER_AGENT`

In [36]:
COLLECTION_NAME = "Quotes"
COLLECTION_DESCRIPTION = """
TODO.
""".strip()

SEED_URLS = [
    "https://quotes.toscrape.com/",
]
ALLOW_DOMAINS = [
    "quotes.toscrape.com",
]
MAX_PAGES = 100

USER_AGENT = "IR Lab Crawler (https://github.com/tira-io/teaching-ir-with-shared-tasks)"

### Step 2: Extract the title and main content from a single page

Before we write the crawler, let's focus on a single page, e.g., the first seed URL. \
During the crawl, Scrapy will download web pages as `HtmlResponse` objects.
Here, we manually download a page and wrap it in the same `HtmlResponse` class, so we can try out exactly the code we will later use in the crawler:

In [5]:
from requests import get
from scrapy.http import HtmlResponse

url = SEED_URLS[0]
response = HtmlResponse(
    url=url,
    body=get(url, headers={"User-Agent": USER_AGENT}, timeout=30).content,
)
response.status

200

#### Extract the title|

HTML pages have a title in the `<title>` tag (which browsers show in the tab). \
With Scrapy, we can select parts of the HTML with [CSS selectors](https://docs.scrapy.org/en/latest/topics/selectors.html) (here, `::text` selects the text inside an element). \
Extra whitespace is removed with `.strip()`. Some pages have no title at all, so our code must handle `None`.

In [10]:
def extract_title(response: HtmlResponse) -> str:
    title = response.css("title::text").get()
    if title is None:
        return ""
    return title.strip()


print(extract_title(response))

Quotes to Scrape


For your crawler, you can also experiment with using the main heading `<h1>` (using `response.css("h1::text").get()`) instead of the `<title>` element.

#### Extract the main content

We do not want to have all the *boilerplate* (e.g., navigation menus, headers, footers, cookie banners, or ads) in our index later on. Instead, we only want to crawl the *main content* of each page.

[Resiliparse](https://resiliparse.chatnoir.eu/en/latest/man/extract/html2text.html) can extract plain text from HTML and remove boilerplate with `main_content=True`.

In [11]:
from resiliparse.extract.html2text import extract_plain_text

def extract_main_content(response: HtmlResponse) -> str:
    return extract_plain_text(response.text, main_content=True)


print(extract_main_content(response))

Quotes to Scrape

Login

“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.” by (about)
Tags: change deep-thoughts thinking world
“It is our choices, Harry, that show what we truly are, far more than our abilities.” by (about)
Tags: abilities choices
“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.” by (about)
Tags: inspirational life live miracle miracles
“The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.” by (about)
Tags: aliteracy books classic humor
“Imperfection is beauty, madness is genius and it's better to be absolutely ridiculous than absolutely boring.” by (about)
Tags: be-yourself inspirational
“Try not to become a man of success. Rather become a man of value.” by (about)
Tags: adulthood success value
“It is better to be hated for what you are than to be loved for what you are no

Note: Boilerplate removal is a heuristic, so check the extracted text for a few pages of your crawled websites to see whether it works well for them.

### Step 3: Assign a stable document ID

Every document needs a unique ID, so that topics and relevance judgments can refer to it later. A good document ID is:

- Unique: two different pages never get the same ID, and
- Stable: the same page always gets the same ID, even if you crawl again or in a different order.

So, instead of counting pages (`doc-1`, `doc-2`, ...), we hash the URL instead.
Python's [BLAKE2](https://docs.python.org/3/library/hashlib.html#blake2) implementation is a good tradeoff between speed and collision resistance.

In [12]:
from hashlib import blake2b


def document_id(url: str) -> str:
    return blake2b(url.encode("utf-8"), digest_size=16).hexdigest()


print(document_id(url))
print(document_id(url))  # Same URL, same ID.
print(document_id("https://quotes.toscrape.com/page/2/"))  # Other URL, other ID.

77c5333b7d4089bc372d620e3635e733
77c5333b7d4089bc372d620e3635e733
7583ca60b1f19e820c71b5ab6c4fe9a3


### Step 4: Find links to follow

A crawler discovers new pages by following the links on the pages it already downloaded. \
Scrapy's `LxmlLinkExtractor` finds all links (`<a href="...">`) on a page, converts relative links to absolute URLs, and removes duplicate URLs. \
Optionally, filter the extracted links with `allow_domains`, to keep only links to the domains we want to crawl.

In [14]:
from scrapy.linkextractors.lxmlhtml import LxmlLinkExtractor

link_extractor = LxmlLinkExtractor(allow_domains=ALLOW_DOMAINS)
links = link_extractor.extract_links(response)
print(f"Found {len(links)} links.")
for link in links[:10]:
    print(link)

Found 47 links.
Link(url='https://quotes.toscrape.com/', text='Quotes to Scrape', fragment='', nofollow=False)
Link(url='https://quotes.toscrape.com/login', text='Login', fragment='', nofollow=False)
Link(url='https://quotes.toscrape.com/author/Albert-Einstein', text='(about)', fragment='', nofollow=False)
Link(url='https://quotes.toscrape.com/tag/change/page/1/', text='change', fragment='', nofollow=False)
Link(url='https://quotes.toscrape.com/tag/deep-thoughts/page/1/', text='deep-thoughts', fragment='', nofollow=False)
Link(url='https://quotes.toscrape.com/tag/thinking/page/1/', text='thinking', fragment='', nofollow=False)
Link(url='https://quotes.toscrape.com/tag/world/page/1/', text='world', fragment='', nofollow=False)
Link(url='https://quotes.toscrape.com/author/J-K-Rowling', text='(about)', fragment='', nofollow=False)
Link(url='https://quotes.toscrape.com/tag/abilities/page/1/', text='abilities', fragment='', nofollow=False)
Link(url='https://quotes.toscrape.com/tag/choices/p

If filtering by domain name does not fit your crawl, check out the other available filtering options: https://docs.scrapy.org/en/latest/topics/link-extractors.html

### Step 5: Combine everything into a Scrapy spider

Now, let's combine everything into a Scrapy spider (a synonym for web crawler). \
We create a class overriding the `Spider` class and define:

- A name for the spider (`name`),
- Seed URLs where the crawl starts (`start_urls`),
- Domains the spider may download pages from (`allowed_domains`)
- A `parse()` method that Scrapy calls for each downloaded page. It can `yield` either a Python dictionary with a document or a `Request` for another URL to visit next.

Scrapy takes care of the rest: it downloads pages in parallel, never downloads the same URL twice, respects `robots.txt`, and saves the documents to a file.

In [17]:
from typing import Iterator

from scrapy import Request, Spider
from scrapy.http import Response


class CollectionSpider(Spider):
    name = "collection"
    start_urls = SEED_URLS
    link_extractor = LxmlLinkExtractor(allow_domains=ALLOW_DOMAINS)

    def parse(self, response: Response) -> Iterator[dict[str, str] | Request]:
        if not isinstance(response, HtmlResponse):
            # Not an HTML page (e.g., an image or a PDF), so there is no text to extract.
            return

        # Save the page as a document (using the helpers above).
        yield {
            "docno": document_id(response.url),
            "url": response.url,
            "title": extract_title(response),
            "text": extract_main_content(response),
        }
        print(f"Saved document {response.url}.")

        # Follow all links on the page.
        for link in self.link_extractor.extract_links(response):
            yield Request(link.url, callback=self.parse)

### Step 6: Run the crawl

Scrapy's [settings](https://docs.scrapy.org/en/latest/topics/settings.html) control how the spider crawls. Most importantly, they make sure our crawler is polite:

- `USER_AGENT`: identify the crawler
- `ROBOTSTXT_OBEY`: respect the website's explicit rules for crawlers, defined in a [`robots.txt`](https://en.wikipedia.org/wiki/Robots.txt) file (Scrapy first downloads the rules and then skips all pages that it disallows for our user agent.)
- `CONCURRENT_REQUESTS` and `DOWNLOAD_DELAY`: Send at most 4 requests at the same time, and wait half a second between requests to the same website.
- `AUTOTHROTTLE_ENABLED`: Slow down automatically if the website responds slowly.
- `HTTPCACHE_ENABLED`: Cache requests, so that running the crawl again does not download the same pages again.
- `CLOSESPIDER_PAGECOUNT`: Stop after roughly `MAX_PAGES` pages (a few more pages may arrive while the crawler shuts down).
- `FEEDS`: Save all crawled documents to the file `crawl.jsonl` (and overwrite the file if it already exists).

The `AsyncCrawlerRunner` runs our spider, and `await` waits until the crawl has finished. Depending on `MAX_PAGES`, this takes a while.
(Ignore the Scrapy download handler warning.)

In [26]:
from scrapy.crawler import AsyncCrawlerRunner
from scrapy.utils.log import configure_logging

settings = {
    "USER_AGENT": USER_AGENT,
    "ROBOTSTXT_OBEY": True,
    "CONCURRENT_REQUESTS": 4,
    "DOWNLOAD_DELAY": 0.5,
    "AUTOTHROTTLE_ENABLED": True,
    "AUTOTHROTTLE_TARGET_CONCURRENCY": 1,
    "HTTPCACHE_ENABLED": True,
    "CLOSESPIDER_PAGECOUNT": MAX_PAGES,
    "FEEDS": {
        "crawl.jsonl": {"format": "jsonlines", "encoding": "utf-8", "overwrite": True},
    },
    # Use Python's built-in `asyncio` instead of Twisted, so that we can run the crawl in a Jupyter notebook.
    "TWISTED_REACTOR_ENABLED": False,
    # Only show warnings and errors. Use "INFO" to see what the crawler is doing.
    "LOG_LEVEL": "WARNING",
}

configure_logging(settings)
runner = AsyncCrawlerRunner(settings)
await runner.crawl(CollectionSpider)

2026-11-02 13:07:20 [scrapy.core.downloader.handlers._base_streaming] WARNING: AiohttpDownloadHandler is experimental and is not recommended for production use.
2026-11-02 13:07:20 [scrapy.core.downloader.handlers._base_streaming] WARNING: AiohttpDownloadHandler is experimental and is not recommended for production use.


Saved document https://quotes.toscrape.com/.
Saved document https://quotes.toscrape.com/tag/simile/.
Saved document https://quotes.toscrape.com/tag/truth/.
Saved document https://quotes.toscrape.com/tag/friends/.
Saved document https://quotes.toscrape.com/tag/friendship/.
Saved document https://quotes.toscrape.com/tag/reading/.
Saved document https://quotes.toscrape.com/tag/books/.
Saved document https://quotes.toscrape.com/tag/humor/.
Saved document https://quotes.toscrape.com/tag/life/.
Saved document https://quotes.toscrape.com/tag/inspirational/.
Saved document http://quotes.toscrape.com/author/Alfred-Tennyson/.
Saved document https://quotes.toscrape.com/tag/inspirational/page/2/.
Saved document https://quotes.toscrape.com/tag/love/.
Saved document https://quotes.toscrape.com/tag/peace/page/1/.
Saved document https://quotes.toscrape.com/page/2/.
Saved document https://quotes.toscrape.com/tag/simile/page/1/.
Saved document http://quotes.toscrape.com/author/Helen-Keller/.
Saved docum

Hint: If your crawl ends without any documents, check whether the website's `robots.txt` disallows your seed URLs, and if so, pick another website.

### Step 7: Inspect the crawled documents

The file `crawl.jsonl` uses the [JSON Lines](https://jsonlines.org/) format: \
Each line is one document as a JSON object with the fields `docno`, `url`, `title`, and `text`. We can load it as a Pandas data frame:

In [27]:
from pandas import read_json

crawl = read_json("crawl.jsonl", lines=True, dtype=str)
print(f"Crawled {len(crawl)} documents.")
crawl.head()

Crawled 101 documents.


,docno,url,title,text
0,77c5333b7d4089bc372d620e3635e733,https://quotes.toscrape.com/,Quotes to Scrape,Quotes to Scrape\n\nLogin\n\n“The world as we ...
1,3c7efc6f5d932ed9a0c2946f4c5f48fc,https://quotes.toscrape.com/tag/simile/,Quotes to Scrape,Quotes to Scrape\n\nLogin\n\nViewing tag: simi...
2,6b35f35e6f37ec8a7511ff8d636596f5,https://quotes.toscrape.com/tag/truth/,Quotes to Scrape,Quotes to Scrape\n\nLogin\n\nViewing tag: trut...
3,c30f505b1290a31c81e77e7b00380ffd,https://quotes.toscrape.com/tag/friends/,Quotes to Scrape,Quotes to Scrape\n\nLogin\n\nViewing tag: frie...
4,fdc9bd8b2cc20262518ac3340d91009f,https://quotes.toscrape.com/tag/friendship/,Quotes to Scrape,Quotes to Scrape\n\nLogin\n\nViewing tag: frie...


Let's check some statistics on the document lengths.

In [28]:
crawl["text_length"] = crawl["text"].str.len()
crawl["text_length"].describe()

count     101.000000
mean      870.128713
std       855.422453
min        23.000000
25%       253.000000
50%       371.000000
75%      1469.000000
max      3505.000000
Name: text_length, dtype: float64

Very short texts are often pages without real content, such as overview or error pages:

In [29]:
crawl.sort_values("text_length").head(5)

,docno,url,title,text,text_length
90,fc876ad020d091ca816824ad44420ac6,http://quotes.toscrape.com/login,Quotes to Scrape,Quotes to Scrape\n\nLogin,23
95,a68b14ae8fde34fff5dda21e7300b284,http://quotes.toscrape.com/tag/romance/page/1/,Quotes to Scrape,Quotes to Scrape\n\nLogin\n\nViewing tag: roma...,209
22,5ca3bc64331221639616881ab181b1ea,https://quotes.toscrape.com/tag/adventure/page/1/,Quotes to Scrape,Quotes to Scrape\n\nLogin\n\nViewing tag: adve...,214
79,f854d721392256be58caa618029ba678,https://quotes.toscrape.com/tag/fear/page/1/,Quotes to Scrape,Quotes to Scrape\n\nLogin\n\nViewing tag: fear...,221
92,5990392f7ee0c4613f41b0d4d1865175,https://quotes.toscrape.com/tag/girls/page/1/,Quotes to Scrape,Quotes to Scrape\n\nLogin\n\nViewing tag: girl...,226


We can also look for duplicates. The same page can be reachable under different URLs (e.g., with and without `index.html`) but still have the same content:

In [30]:
print("Duplicate docnos:", crawl["docno"].duplicated().sum())
print("Duplicate texts:", crawl["text"].duplicated().sum())

Duplicate docnos: 0
Duplicate texts: 5


### Step 8: Clean up the documents

After crawling, we can filter the documents. \
Which filters make sense depends on your crawled websites, so look at your data and adapt the filters below. \
For documentation, we note every filter in `filtering_steps` (you enter such details when submitting your collection).

In this example, we:
1. Remove documents with very short texts (e.g., login or error pages),
2. Remove documents whose text is identical to another document, and
3. Remove duplicate document IDs (should not occur, as Scrapy de-duplicates URLs).

In [31]:
MIN_TEXT_LENGTH = 200

documents = crawl.copy()
filtering_steps = []

documents = documents[documents["text_length"] >= MIN_TEXT_LENGTH]
filtering_steps.append(f"Removed documents with less than {MIN_TEXT_LENGTH} characters of main content text.")

documents = documents.drop_duplicates(subset="text")
filtering_steps.append("Removed documents with duplicate text.")

documents = documents.drop_duplicates(subset="docno")

print(f"Kept {len(documents)} of {len(crawl)} documents.")
documents.head()

Kept 95 of 101 documents.


,docno,url,title,text,text_length
0,77c5333b7d4089bc372d620e3635e733,https://quotes.toscrape.com/,Quotes to Scrape,Quotes to Scrape\n\nLogin\n\n“The world as we ...,1469
1,3c7efc6f5d932ed9a0c2946f4c5f48fc,https://quotes.toscrape.com/tag/simile/,Quotes to Scrape,Quotes to Scrape\n\nLogin\n\nViewing tag: simi...,403
2,6b35f35e6f37ec8a7511ff8d636596f5,https://quotes.toscrape.com/tag/truth/,Quotes to Scrape,Quotes to Scrape\n\nLogin\n\nViewing tag: trut...,645
3,c30f505b1290a31c81e77e7b00380ffd,https://quotes.toscrape.com/tag/friends/,Quotes to Scrape,Quotes to Scrape\n\nLogin\n\nViewing tag: frie...,1655
4,fdc9bd8b2cc20262518ac3340d91009f,https://quotes.toscrape.com/tag/friendship/,Quotes to Scrape,Quotes to Scrape\n\nLogin\n\nViewing tag: frie...,873


### Step 9: Save the documents

Now, save again the (filtered) document collection as a JSON Lines file named `documents.jsonl`, keeping only the important fields: `docno` (unique, stable document ID), `url`, `title`, `text` (main content).

In [33]:
documents[["docno", "url", "title", "text"]].to_json("documents.jsonl", orient="records", lines=True)

### Step 10: Submit your document collection

Your document collection is ready. To submit it, fill in the course's document collection submission form and upload your `documents.jsonl` file. The form asks for the name and description of your collection (from the start of this tutorial), your seed URLs, and how you filtered the documents.

As a recap, your submission should include:

In [39]:
print("Title:", COLLECTION_NAME)
print("Description:\n", COLLECTION_DESCRIPTION)
print("Seed URLs:", ", ".join(SEED_URLS))
print("Additional filtering/post-processing:\n", " ".join(filtering_steps))

Title: Quotes
Description:
 TODO.
Seed URLs: https://quotes.toscrape.com/
Additional filtering/post-processing:
 Removed documents with less than 200 characters of main content text. Removed documents with duplicate text.


Your teachers then download all submitted collections and bundle them with `teaching-ir convert-documents-csv` into one large document collection for the whole course.

### Your turn

Now adapt this notebook to your own topic:

1. Plan your collection and change the variables in Step 1. Look at a few pages by hand: Is the title meaningful? Does the main content extraction work? Which links does the crawler follow?
2. Run a small crawl (`MAX_PAGES = 100`), inspect the results, and adapt the filters.
3. If everything looks good, increase `MAX_PAGES` for your final crawl.

Some ideas to improve your crawler:
- Skip unwanted pages early with the link extractor's `deny=` option instead of filtering them afterwards.
- Limit how far the crawler goes from the seed URLs with the [`DEPTH_LIMIT`](https://docs.scrapy.org/en/latest/topics/settings.html#depth-limit) setting.
- Only keep documents in a specific language, e.g., with Resiliparse's [language detection](https://resiliparse.chatnoir.eu/en/latest/man/parse/lang.html).